# 05 — Goal 4: perceptual family alignment

Audits four independent RA annotations, evaluates matched family alignment, and compares the 4RA and merged 2RA systems on shared recordings.

Every displayed denominator and paper-facing visual is also saved under `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside paper1_pipeline_rebuilt.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("Visualization outputs:", VIZ_ROOT)


Primary alignment excludes competing speech and non-task content because the estimand is family perceptual alignment, not source recognition. The broad metadata direction gate must be confirmed from the RA codebook before the 4RA-versus-2RA comparison runs.

In [ ]:
RUN_GOAL_4 = False
if RUN_GOAL_4:
    run_cli("human-qc", "--schema", "config/human_qc_schema.yaml")
else:
    print("Using existing Goal 4 outputs.")


In [ ]:
rating_coverage = read_stage("04_analysis/human_qc/rating_design_item_coverage")
design_summary = read_stage("04_analysis/human_qc/rating_design_summary")
ratings = read_stage("04_analysis/human_qc/ratings_long")
agreement = read_stage("04_analysis/human_qc/interrater_agreement")
consensus = read_stage("04_analysis/human_qc/four_ra_consensus_primary")
direction_audit = read_stage("04_analysis/human_qc/two_ra_broad_direction_and_scale_audit")

display(direction_audit)
assert direction_audit["direction"].eq("higher_is_worse").all()
save_table(direction_audit, "05_goal4", "direction_and_scale_audit")
save_table(design_summary, "05_goal4", "four_ra_design_summary")
display(design_summary)


In [ ]:
# Coverage heatmap makes missing/rotating rater designs visible.
coverage_matrix = (
    ratings.assign(rated=1)
    .pivot_table(index=["file_name", "category"], columns="rater_id", values="rated", aggfunc="max", fill_value=0)
)
save_table(coverage_matrix.reset_index(), "05_goal4", "four_ra_coverage_matrix")
fig, ax = plt.subplots(figsize=(10, min(18, max(5, .08 * len(coverage_matrix)))))
sns.heatmap(coverage_matrix, cmap=["#f2f2f2", "#4C78A8"], cbar=False, ax=ax)
ax.set(title="Independent detailed-rating coverage", xlabel="Rater", ylabel="Recording × perceptual family")
ax.tick_params(axis="y", labelleft=False)
save_figure(fig, "05_goal4", "four_ra_rating_coverage")
plt.show()


In [ ]:
# Prevalence is shown beside agreement because sparse categories can distort kappa.
prevalence = (
    consensus.groupby("category")["consensus_rating"]
    .agg(
        n_consensus="count",
        positive=lambda x: int(pd.to_numeric(x, errors="coerce").sum()),
        prevalence=lambda x: float(pd.to_numeric(x, errors="coerce").mean()),
    ).reset_index()
)
agreement_view = agreement.merge(prevalence, on="category", how="left")
save_table(agreement_view, "05_goal4", "agreement_and_prevalence")
display(agreement_view)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=prevalence, x="prevalence", y="category", ax=axes[0], color="#4C78A8")
axes[0].set(title="4RA consensus artifact prevalence", xlabel="Positive fraction", ylabel="")
axes[1].errorbar(
    agreement_view["gwet_ac1_nominal"],
    np.arange(len(agreement_view)),
    xerr=np.vstack([
        agreement_view["gwet_ac1_nominal"] - agreement_view["gwet_ac1_ci_low"],
        agreement_view["gwet_ac1_ci_high"] - agreement_view["gwet_ac1_nominal"],
    ]),
    fmt="o", color="0.2", ecolor="0.55", capsize=3,
)
axes[1].set_yticks(np.arange(len(agreement_view)), agreement_view["category"])
axes[1].set(title="Gwet AC1 with item-bootstrap 95% CI", xlabel="Agreement", xlim=(-.1, 1.05))
fig.tight_layout()
save_figure(fig, "05_goal4", "prevalence_and_agreement")
plt.show()


In [ ]:
# Primary cross-family matrix. Diagonal cells are matched perceptual families.
four = read_stage("04_analysis/human_qc/four_ra_family_alignment_matrix")
four_effect = four.pivot(index="human_family", columns="objective_family", values="effect")
four_n = four.pivot(index="human_family", columns="objective_family", values="n_recordings")
save_table(four, "05_goal4", "four_ra_family_alignment_matrix")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(four_effect, vmin=-1, vmax=1, center=0, cmap="vlag", annot=True, fmt=".2f", ax=axes[0])
axes[0].set(title="4RA family alignment effect", xlabel="Objective Q family", ylabel="Perceptual family")
sns.heatmap(four_n, cmap="viridis", annot=True, fmt=".0f", ax=axes[1])
axes[1].set(title="Pair-specific recording denominator", xlabel="Objective Q family", ylabel="")
fig.tight_layout()
save_figure(fig, "05_goal4", "four_ra_alignment_and_denominators")
plt.show()

matched_summary = (
    four.loc[four["estimable"]]
    .groupby("matched_family")["effect"]
    .agg(["count", "mean", "median"]).reset_index()
)
save_table(matched_summary, "05_goal4", "matched_vs_mismatched_descriptive")
display(matched_summary)

specificity = read_stage("04_analysis/human_qc/four_ra_matched_family_specificity")
save_table(specificity, "05_goal4", "four_ra_matched_family_specificity")
display(specificity)


In [ ]:
# The merged 2RA workflow is comparable only for families with explicit overlap.
comparison = read_stage("04_analysis/human_qc/four_ra_vs_two_ra_paired_alignment")
save_table(comparison, "05_goal4", "four_ra_vs_two_ra_paired_alignment")
display(comparison)

if "delta_auc_a_minus_b" in comparison and comparison["delta_auc_a_minus_b"].notna().any():
    plot = comparison.loc[comparison["status"].eq("ok")].copy()
    fig, ax = plt.subplots(figsize=(9, max(4, .8 * len(plot))))
    ax.errorbar(
        plot["delta_auc_a_minus_b"],
        np.arange(len(plot)),
        xerr=np.vstack([
            plot["delta_auc_a_minus_b"] - plot["delta_ci_low"],
            plot["delta_ci_high"] - plot["delta_auc_a_minus_b"],
        ]),
        fmt="o", capsize=3, color="0.2", ecolor="0.55",
    )
    ax.axvline(0, color="0.35", linestyle="--")
    ax.set_yticks(np.arange(len(plot)), plot["family"])
    ax.set(
        title="Paired shared-recording comparison",
        xlabel="ΔAUC: 4RA detailed − merged 2RA broad",
        ylabel="",
    )
    save_figure(fig, "05_goal4", "four_ra_minus_two_ra_delta_auc")
    plt.show()


In [ ]:
# Secondary duration/fraction analysis preserves the richer interval annotations.
extent = read_stage("04_analysis/human_qc/four_ra_extent_consensus_secondary")
context = read_stage("04_analysis/human_qc/context_annotations_not_family_alignment")
save_table(extent, "05_goal4", "four_ra_extent_consensus_secondary")

extent_summary = (
    extent.groupby("category")["consensus_annotated_fraction"]
    .agg(n="count", median="median", q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75))
    .reset_index()
)
save_table(extent_summary, "05_goal4", "extent_consensus_summary")
display(extent_summary)

context_summary = (
    context.groupby("category")["rating"]
    .agg(rater_recordings="size", positive="sum", prevalence="mean")
    .reset_index()
)
context_summary["primary_family_alignment"] = False
save_table(context_summary, "05_goal4", "context_annotations_excluded_from_family_alignment")
display(context_summary)
